In [3]:
import pandas as pd
import json
import requests
from bs4 import BeautifulSoup
from io import StringIO

# Function to fetch and parse timezone data
def fetch_timezone_data(url):
    response = requests.get(url)
    if response.status_code != 200:
        raise ValueError(f"Failed to fetch page: {response.status_code}")
    
    # Parse the page content with BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')
    html_tables = str(soup.find_all('table'))  # Extract all tables as HTML strings
    tables = pd.read_html(StringIO(html_tables), header=[0, 1])  # Parse tables with multi-level headers

    for table in tables:
        # Check if the table contains the relevant columns
        if ('TZ identifier' in table.columns.get_level_values(1) and
            'UTC offset ±hh:mm' in table.columns.get_level_values(0)):
            # Extract relevant columns using multi-level headers
            timezone_data = table[[('TZ identifier', 'TZ identifier'), 
                                   ('UTC offset ±hh:mm', 'SDT')]].copy()
            timezone_data.columns = ['timezone', 'utc_offset']  # Rename columns
            return timezone_data.to_dict(orient='records')  # Convert to list of dicts
    
    raise ValueError("Expected timezone table not found.")

# Wikipedia URL for the timezone database
wikipedia_url = "https://en.wikipedia.org/wiki/List_of_tz_database_time_zones"

# Fetch, process, and save the data
try:
    timezone_data = fetch_timezone_data(wikipedia_url)
    # Save the data to a JSON file
    with open("timezone_data.json", "w", encoding="utf-8") as file:
        json.dump(timezone_data, file, indent=4)
    
    print("Timezone data has been saved to 'timezone_data.json'")
except Exception as e:
    print(f"An error occurred: {e}")


Timezone data has been saved to 'timezone_data.json'
